# feax4d Quickstart — 4D-printing design pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naruki-Ichihara/feax4d/blob/main/examples/colab_quickstart.ipynb)

End-to-end **4D-printing** pipeline built on [FEAX](https://github.com/Naruki-Ichihara/feax):

1. **Optimise** a two-layer thermal shell so a clamped cantilever *stays flat* under a free-edge load — the cooling-induced bilayer warping cancels the load deflection (density + fibre orientation are the design variables).
2. **Fibre paths** — relax an aligned Swift–Hohenberg stripe field along the optimised per-layer fibre director and extract fibre-path polylines.
3. **G-code** — fill each layer's **non-fibre** region with polymer infill (using the optimised density distribution) and emit Fibrifier (9T Labs) G-code.

Tested on a free Colab runtime (CPU and T4 GPU). First-time setup takes ~5–8 minutes (one-off, dominated by the SuiteSparse build); the demo pipeline below runs in ~1–3 minutes at the small demo resolution.

## 1. Install

`feax4d` is installed from GitHub. It builds on **FEAX** (the FE backend), so we first run FEAX's Colab setup script (gmsh, NLopt, BLAS/LAPACK, SuiteSparse) and install FEAX exactly as in the FEAX quickstart, then add `feax4d` and its only extra runtime dependency, `shapely` (polymer-infill region geometry).

`feax4d` is installed with `--no-deps` because its `feax` dependency is already satisfied by the git install above (it is not on PyPI).

In [ ]:
!curl -fsSL https://raw.githubusercontent.com/Naruki-Ichihara/feax/main/scripts/colab_setup.sh | bash
!SUITESPARSE_INCLUDE_DIR=/usr/local/include/suitesparse \
 SUITESPARSE_LIBRARY_DIR=/usr/local/lib \
 pip install -q "feax[cuda13,sksparse] @ git+https://github.com/Naruki-Ichihara/feax.git"
!CMAKE_ARGS="-DBUILD_PBATCH_SOLVE=OFF" pip install -q --no-build-isolation git+https://github.com/johnviljoen/spineax.git
# feax4d + its only extra runtime dep (feax / scikit-image / meshio / matplotlib / nlopt already installed above)
!pip install -q shapely
!pip install -q --no-deps git+https://github.com/Naruki-Ichihara/feax4d.git

In [ ]:
import jax
import feax as fe
import feax4d

print('JAX version   :', jax.__version__)
print('FEAX version  :', fe.__version__)
print('feax4d version:', feax4d.__version__)
print('Backend       :', jax.default_backend())
print('Devices       :', jax.devices())

## 2. Optimise the stable plate

A cantilever clamped on its **right** edge (span : width = 2 : 1) carries a downward line load on the free (left) edge. The optimiser tailors the per-layer density and fibre orientation so the one-shot cooling ($\Delta T = -150$ K) warps the bilayer just enough to cancel the load deflection — i.e. the plate **stays flat**:

$$ J(\rho, \theta) \;=\; \frac{\lVert w \rVert^2}{\lVert w_{\text{init}} \rVert^2} \;\longrightarrow\; 0 .$$

We use a small demo mesh (40 × 20) and 40 iterations so it finishes quickly; raise `Nx, Ny, max_iter` for a production run.

In [ ]:
from pathlib import Path
import feax4d

OUT = Path('output')
cfg = feax4d.OptimizeConfig(
    Lx=200e-3, Ly=100e-3, Nx=40, Ny=20,   # demo resolution (raise for production)
    clamp='right', load_mag=5.0, delta_t=-150.0,
    target_fn=None,                        # None => flat target (stay-flat objective)
    max_iter=40,
    output_dir=OUT / 'stable_plate',
)
result = feax4d.optimize(cfg)
print('\nstop reason :', result.stop_reason)
print('best obj    : %.4e @ iter %d' % (result.best_obj, result.best_iter))
print('history     :', result.xdmf_path)

In [ ]:
import matplotlib.pyplot as plt

h = result.history
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(h['iter'], h['obj'], 'b-')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel(r'$\|w\|^2 / \|w_{init}\|^2$')
ax[0].set_title('stay-flat objective'); ax[0].grid(alpha=0.3)
ax[1].plot(h['iter'], h['vol'], 'g-')
ax[1].set_xlabel('iteration'); ax[1].set_ylabel('mean fibre fraction')
ax[1].set_title('mean density (diagnostic)'); ax[1].set_ylim(0, 1); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Fibre paths (aligned Swift–Hohenberg)

For each layer we relax an aligned-SH stripe field along the optimised fibre director (masked to the fibre region) and extract the stripe centre-lines as fibre-path polylines. Outputs (SVG / NPZ / VTU / PNG) are written under `output/stable_plate/fibre_paths/`.

In [ ]:
paths = feax4d.generate_fibre_paths(
    xdmf_path=result.xdmf_path,
    stripe_period=4.0e-3,    # fibre / tow spacing [m]
    refine_factor=3,         # mesh refinement for the SH relaxation
    n_steps=300,
    rho_cutoff=0.5,
)
fibre_dir = result.xdmf_path.parent / 'fibre_paths'
print('fibre paths per layer:', {k: len(v) for k, v in paths.items()})

In [ ]:
from IPython.display import Image, display

# Aligned-SH stripe rasters (fibre direction follows the optimised director).
for layer in (0, 1):
    png = fibre_dir / f'sh_stripe_layer{layer}.png'
    if png.exists():
        display(Image(filename=str(png)))

## 4. Polymer-filled G-code

Each design layer becomes a **polymer-infill layer (P)** followed by a **fibre layer (F)**. The polymer region is taken from the optimised **density distribution** — polymer fills only the *non-fibre* part of each layer (the fibre region is excluded), so polymer does not intrude into the fibre tows.

In [ ]:
params = feax4d.FibrifierParams()
params.temperature.bed_temperature = 90      # deg C
params.layer_height = 0.15                   # mm

gres = feax4d.fibre_paths_to_gcode(
    fibre_dir,
    params=params,
    polymer_fill=True,            # fill non-fibre regions with polymer (density-limited)
    infill_angle=[0.0, 90.0],     # per-layer polymer scan direction
    infill_pitch=1.0,             # polymer line spacing [mm]
    connection_threshold=10.0,    # merge fibre path ends within 10 mm
)
print('\ngcode        :', gres['gcode_path'])
print('layers       :', gres['n_layers'])
print('fibre paths  :', gres['n_fiber_paths'])
print('polymer paths:', gres['n_polymer_paths'])
print('total fibre  : %.0f mm' % gres['total_fiber_mm'])

### Per-layer preview

One preview per layer: fibre paths (coloured) + polymer infill. Note the polymer infill stops at the fibre boundary — it is limited to the non-fibre region by the optimised density.

In [ ]:
from IPython.display import Image, display

for pv in gres['preview_paths']:
    display(Image(filename=pv))

### Peek at the generated G-code

In [ ]:
with open(gres['gcode_path']) as f:
    head = f.readlines()[:30]
print(''.join(head))

## Next steps

- Raise `Nx, Ny, max_iter` (Section 2) and `n_steps, refine_factor` (Section 3) for production-quality results.
- Shape matching instead of stay-flat: pass `target_fn=lambda x, y: ...` to `OptimizeConfig` (returns the desired transverse displacement `w_target(x, y)` at the mesh nodes).
- Tune the print with `feax4d.FibrifierParams` (temperatures, speeds, offsets, fibre/extrusion settings).
- Reload a saved design with `feax4d.load_design('output/stable_plate')` and the run summary with `feax4d.load_summary(...)`.

Repository: <https://github.com/Naruki-Ichihara/feax4d>  ·  FE backend: <https://github.com/Naruki-Ichihara/feax>